# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 2: KỸ THUẬT ĐẶC TRƯNG & THIẾT LẬP PIPELINE (FEATURE ENGINEERING & PIPELINE)

```text
[clean_emails.csv] ──> Mục 4 (FE Biến đổi & Domain: Log, Binning, caps_ratio, !, $)
                   ──> Mục 5 (FE Lọc & Pipeline: Chia Train/Test chống Leak, TF-IDF, Chi2)
                   ──> [train_test_data.joblib & preprocessor.joblib]
```


### 1.1 Khai báo Thư viện & Nạp Dữ liệu Sạch từ Notebook 1
Ở giai đoạn Kỹ thuật đặc trưng (Feature Engineering) và Xây dựng Pipeline, ta sử dụng các thư viện chuyển đổi dữ liệu của `scikit-learn` (`TfidfVectorizer`, `SelectKBest`, `chi2`, `Pipeline`) cùng công cụ tuần tự hóa `joblib` để lưu trữ mô hình và dữ liệu trung gian.
Ta tiến hành nạp tệp dữ liệu đã làm sạch `clean_emails.csv` từ thư mục `../data/processed/`.

In [ ]:
# Khai báo thư viện & Nạp dữ liệu sạch từ Notebook 1
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CLEAN_DATA_PATH = '../data/processed/clean_emails.csv'
if not os.path.exists(CLEAN_DATA_PATH):
    raise FileNotFoundError(f'❌ Chưa tìm thấy {CLEAN_DATA_PATH}. Bạn cần chạy hoàn tất Notebook 1 trước!')

df = pd.read_csv(CLEAN_DATA_PATH)
df['text_clean'] = df['text_clean'].fillna('')
print(f'✅ Đã nạp thành công dữ liệu sạch từ Notebook 1: {df.shape[0]:,} dòng, {df.shape[1]} cột')

**Nhận xét:**
- Toàn bộ 39.154 dòng dữ liệu sạch từ Giai đoạn 1 đã sẵn sàng để trích xuất đặc trưng.
- Dữ liệu hoàn toàn không còn giá trị rỗng (`NaN`) ở các trường văn bản cốt lõi.

---
## Mục 4: Kỹ thuật tạo đặc trưng mới & Biến đổi dữ liệu (Feature Engineering & Transformation)

### 2.1 Kỹ thuật Trích xuất Đặc trưng Đa tầng & Biến đổi Dữ liệu
Dựa trên các phát hiện thực nghiệm từ Bảng đúc kết EDA ở Notebook 1 và bám sát yêu cầu đề bài của TS. Nguyễn Thị Khánh Tiên, ta tiến hành tạo 4 nhóm đặc trưng:
1. **Đặc trưng độ dài văn bản:** Tính độ dài tiêu đề (`subject_len`) và nội dung (`text_len`).
2. **Biến đổi Logarithm (`np.log1p`):** Vì phân phối độ dài có đuôi rất dài sang bên phải (lệch phải nặng), ta áp dụng hàm $\ln(1 + x)$ để nén biên độ dao động của các giá trị ngoại lai, đưa phân phối về gần dạng phân phối chuẩn.
3. **Phân nhóm độ dài (Binning):** Phân chia văn bản thành các khoảng kích thước rời rạc (`0-50`, `50-100`, `100-200`, `200-500`, `500-1000`, `1000+`) giúp mô hình nắm bắt phi tuyến tính.
4. **Đặc trưng tri thức miền (Domain Features):**
   - `caps_ratio`: Tỷ lệ ký tự viết hoa trong văn bản gốc (thư spam thường gào thét bằng chữ IN HOA).
   - `exclamation_count`: Số lượng dấu chấm than (`!`) — dấu hiệu thúc giục giật gân.
   - `dollar_count`: Số lượng ký hiệu tiền tệ (`$`) — dấu hiệu mồi chài tài chính trúng thưởng.

In [ ]:
# Tạo đặc trưng độ dài, biến đổi log, binning và các đặc trưng domain
# text_clean được dùng cho độ dài và đã lowercase/loại dấu câu, nên đặc trưng ký tự
# cần được trích xuất từ subject + body gốc.
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['text_len'] = df['text_clean'].fillna('').astype(str).str.len()
df['log_text_len'] = np.log1p(df['text_len'])
df['log_subject_len'] = np.log1p(df['subject_len'])

bins = [0, 50, 100, 200, 500, 1000, float('inf')]
labels = ['0-50', '51-100', '101-200', '201-500', '501-1000', '1000+']
df['text_len_bin'] = pd.cut(df['text_len'], bins=bins, labels=labels, right=False)
df['text_len_bin_code'] = pd.Categorical(df['text_len_bin'], categories=labels, ordered=True).codes

raw_text_series = (
    df['subject'].fillna('').astype(str) + ' ' + df['body'].fillna('').astype(str)
)
raw_text_len = raw_text_series.str.len()
df['caps_count'] = raw_text_series.str.count(r'[A-Z]')
df['caps_ratio'] = (df['caps_count'] / raw_text_len.replace(0, np.nan)).fillna(0.0)
df['exclamation_count'] = raw_text_series.str.count('!')
df['dollar_count'] = raw_text_series.str.count(r'\$')

print('✅ Đã tạo các đặc trưng: text_len, log_text_len, text_len_bin, caps_ratio, exclamation_count, dollar_count')
print(df[['text_len', 'log_text_len', 'caps_ratio', 'exclamation_count', 'dollar_count']].describe().round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['text_len'], bins=50, kde=True, ax=axes[0])
axes[0].set_title(f"text_len gốc (skew={df['text_len'].skew():.2f})")
axes[0].set_xlabel('text_len')

sns.histplot(df['log_text_len'], bins=50, kde=True, ax=axes[1])
axes[1].set_title(f"log1p(text_len) (skew={df['log_text_len'].skew():.2f})")
axes[1].set_xlabel('log1p(text_len)')

plt.tight_layout()
plt.show()

**Đúc kết từ bước Kỹ thuật Đặc trưng:**
- Phép biến đổi Logarithm đã nén thành công độ lệch chuẩn của các văn bản siêu dài mà không làm mất thông tin hay phải loại bỏ bản ghi ngoại lai.
- Các biến domain (`exclamation_count`, `dollar_count`, `caps_ratio`) phản ánh chính xác hành vi tâm lý của những kẻ gửi thư rác, cung cấp tín hiệu phân loại bổ trợ cực kỳ giá trị bên cạnh từ điển TF-IDF.

---
## Mục 5: Thiết lập Pipeline tiền xử lý & Phân chia dữ liệu (Data Preprocessing Pipeline)

### 3.1 Phân chia Tập Huấn luyện & Kiểm tra theo Tỷ lệ Phân tầng (Stratified Train/Test Split)
**Nguyên tắc vàng chống rò rỉ dữ liệu (Zero Data Leakage):**
Ta bắt buộc phải tách tập kiểm tra `X_test, y_test` **TRƯỚC KHI** thực hiện bất kỳ phép biến đổi vector nào (như `fit` TF-IDF hay chọn lọc Chi-square).
- Tỷ lệ chia: 80% tập Train (31.323 mẫu) và 20% tập Test (7.831 mẫu).
- Áp dụng `stratify=y` để bảo đảm tỷ lệ giữa thư rác (Spam ~55.8%) và thư thường (Ham ~44.2%) hoàn toàn đồng nhất giữa cả hai tập.

In [ ]:
# Viết code phân chia Train/Test 80/20 phân tầng (stratify=y, random_state=42) chống rò rỉ dữ liệu tại đây
X = df['text_clean'].fillna('')
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("\nTỷ lệ nhãn Train:")
print(y_train.value_counts(normalize=True))
print("\nTỷ lệ nhãn Test:")
print(y_test.value_counts(normalize=True))

**Nhận xét:**
- Tập Train và Test đã được phân chia cân đối tuyệt đối về tỷ lệ nhãn.
- Dữ liệu `X_train` và `X_test` được lưu ở dạng chuỗi văn bản gốc, hoàn toàn độc lập với mọi quá trình fit thống kê sau này.

### 3.2 Xây dựng Pipeline Tiền xử lý Cơ sở (TF-IDF & Chi-Square SelectKBest)
Văn bản sau khi làm sạch có không gian từ vựng lên đến hàng chục nghìn từ khác nhau, dẫn đến "Lời nguyền số chiều" (**Curse of Dimensionality**). Ta thiết kế Pipeline tiền xử lý gồm 2 giai đoạn:
1. `TfidfVectorizer`: Chuyển đổi văn bản thành ma trận trọng số TF-IDF, khai thác cả từ đơn và cụm 2 từ liền kề (`ngram_range=(1, 2)`).
2. `SelectKBest(chi2)`: Sử dụng kiểm định Chi-bình phương (${\chi}^2$) để đo lường mức độ phụ thuộc giữa từng từ khóa và biến mục tiêu Spam/Ham, chắt lọc ra $k$ từ khóa có sức mạnh phân loại lớn nhất.

*Lưu ý:* Pipeline này được giữ ở trạng thái **chưa fit** (`unfitted`) để GridSearchCV ở Notebook 3 tự động dò tìm siêu tham số tối ưu bên trong từng fold huấn luyện.

In [ ]:
# Thiết lập Pipeline Tiền xử lý (Baseline khởi tạo ban đầu)
# Các siêu tham số max_features và k sẽ được GridSearchCV dò tìm tối ưu tại Notebook 3 (Mục 6)
preprocessor = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('select', SelectKBest(
        score_func=chi2,
        k=1000
    ))
])

print('Đã khởi tạo Pipeline tiền xử lý cơ sở.')
print('Chưa fit TF-IDF và SelectKBest.')

**Nhận xét:**
- Pipeline cơ sở đã được khởi tạo thành công với cấu trúc liên kết 2 tầng: `tfidf` $\rightarrow$ `select`.
- Việc tích hợp bộ chọn lọc Chi-square giúp giảm bớt số chiều không gian đặc trưng, giảm độ phức tạp tính toán và chống quá khớp (overfitting) hiệu quả.

### 3.3 Tuần tự hóa & Lưu trữ Dữ liệu Trung gian Phục vụ Huấn luyện
Để chuyển giao tài nguyên cho Giai đoạn 3 (Huấn luyện Mô hình ở Notebook 3), ta tuần tự hóa các đối tượng thành tệp nhị phân `.joblib`:
1. `../data/processed/train_test_data.joblib`: Lưu trữ từ điển chứa 4 đối tượng `X_train, X_test, y_train, y_test`.
2. `../models/preprocessor.joblib`: Lưu trữ Pipeline tiền xử lý cơ sở.

In [ ]:
# Tạo thư mục lưu dữ liệu và mô hình
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

train_test_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}

joblib.dump(
    train_test_data,
    '../data/processed/train_test_data.joblib'
)

joblib.dump(
    preprocessor,
    '../models/preprocessor.joblib'
)

print("Đã lưu train_test_data.joblib")
print("Đã lưu ../models/preprocessor.joblib")

**Tổng kết Giai đoạn 2:**
- Toàn bộ các đặc trưng nâng cao và bộ khung tiền xử lý đã được chuẩn hóa và đóng gói an toàn.
- Hệ thống dữ liệu đã sẵn sàng 100% để chuyển giao cho Khôi Nguyên (Mục 6) và Đông (Mục 7) thực hiện huấn luyện đa mô hình và kiểm định hiệu năng tại Notebook 3.